# CreditGraph: Stress Test Analysis

I built a credit risk knowledge graph to explore thesis about why topological databases has extra dimension for analysis.

Traditional credit analysis treats each loan as an isolated event. Client A's probability of default has nothing to do with Client B's. But in practice, clients guarantee each other's loans, share company directorships, and belong to the same corporate groups. Their fates are entangled.

This notebook connects to a live Neo4J graph database containing 300 individual clients, 95 companies, and 458 loans -- all connected by ownership, guarantee, and corporate hierarchy relationships. The data is synthetic (built on real UK beneficial ownership topology mapped to Mexican identities), but the structural patterns are keep: they come from how actual companies are owned and controlled (or at leas what we can constructed alike from public info, I guess this relationship are more complex in real life data).

I want to answer one question: **what does this portfolio look like when you stop treating loans as rows in a table and start treating them as nodes in a network?**

**A note on honesty:** The numbers below are not auditable facts. The report demonstrates a method, not a conclusion. Every finding should be read as: "this kind of exposure is invisible to SQL, and here's the query that finds it." A complete educational exercise.

---

In [1]:
# Connection setup
import os
from neo4j import GraphDatabase
import pandas as pd

# Credentials from .env file (not committed to git)
# To run this notebook, create a .env file with:
#   NEO4J_URI=neo4j+s://your-instance.databases.neo4j.io
#   NEO4J_USERNAME=your-username
#   NEO4J_PASSWORD=your-password

URI = os.environ.get("NEO4J_URI", "neo4j+s://1c09a89b.databases.neo4j.io")
USERNAME = os.environ.get("NEO4J_USERNAME", "1c09a89b")
PASSWORD = os.environ.get("NEO4J_PASSWORD", "")

if not PASSWORD:
    # Try loading from .env file in project root
    env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
    if os.path.exists(env_path):
        for line in open(env_path):
            if "=" in line and not line.startswith("#"):
                key, val = line.strip().split("=", 1)
                os.environ[key] = val
        PASSWORD = os.environ.get("NEO4J_PASSWORD", "")

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
driver.verify_connectivity()

def query(cypher):
    """Run a Cypher query and return results as a pandas DataFrame."""
    with driver.session() as session:
        result = session.run(cypher)
        return pd.DataFrame([dict(record) for record in result])

print("Connected to AuraDB")

Connected to AuraDB


---
## Section 1: What the Portfolio Looks Like (the SQL view)

Let's start with the standard analysis, the view any risk analyst would produce from a loan table with a few JOINs. Total exposure, NPL ratio, sector breakdown. This is the baseline(using info that's on web): what the institution already knows about itself.

In [2]:
# Portfolio overview: what a SQL analyst would report
overview = query("""
    MATCH (p:Prestamo)
    RETURN
        count(p) AS total_loans,
        round(sum(p.saldo_vigente_mxn)) AS total_exposure_mxn,
        round(avg(p.saldo_vigente_mxn)) AS avg_loan_mxn,
        sum(CASE WHEN p.estatus = 'VENCIDO' THEN 1 ELSE 0 END) AS vencidos,
        sum(CASE WHEN p.estatus = 'ACTIVO' THEN 1 ELSE 0 END) AS activos,
        sum(CASE WHEN p.estatus = 'REESTRUCTURADO' THEN 1 ELSE 0 END) AS reestructurados,
        sum(CASE WHEN p.estatus = 'PAGADO' THEN 1 ELSE 0 END) AS pagados
""")
print("=== PORTFOLIO OVERVIEW ===")
print(overview.to_string(index=False))

total_loans = overview['total_loans'].iloc[0]
vencidos = overview['vencidos'].iloc[0]
print(f"\nNPL ratio: {vencidos}/{total_loans} = {vencidos/total_loans*100:.1f}%")

=== PORTFOLIO OVERVIEW ===
 total_loans  total_exposure_mxn  avg_loan_mxn  vencidos  activos  reestructurados  pagados
         458         200673473.0      438152.0        36      322               28       72

NPL ratio: 36/458 = 7.9%


In [3]:
# Exposure by entity type
by_type = query("""
    MATCH (p:Prestamo)
    WHERE p.estatus IN ['ACTIVO', 'REESTRUCTURADO']
    RETURN p.tipo_titular AS tipo,
           count(p) AS num_loans,
           round(sum(p.saldo_vigente_mxn)) AS exposure
    ORDER BY exposure DESC
""")
print("=== EXPOSURE BY ENTITY TYPE ===")
print(by_type.to_string(index=False))

=== EXPOSURE BY ENTITY TYPE ===
      tipo  num_loans    exposure
   EMPRESA        118 152194124.0
INDIVIDUAL        232  27829451.0


In [4]:
# Exposure by sector (corporate loans only)
by_sector = query("""
    MATCH (e:Empresa)-[:TIENE_PRESTAMO]->(p:Prestamo)
    WHERE p.estatus IN ['ACTIVO', 'REESTRUCTURADO']
    RETURN e.sector AS sector,
           count(p) AS num_loans,
           round(sum(p.saldo_vigente_mxn)) AS exposure
    ORDER BY exposure DESC
""")
print("=== CORPORATE EXPOSURE BY SECTOR ===")
print(by_sector.to_string(index=False))

=== CORPORATE EXPOSURE BY SECTOR ===
      sector  num_loans   exposure
   SERVICIOS         38 52423593.0
 MANUFACTURA         34 46136178.0
    COMERCIO         30 35188252.0
CONSTRUCCION         11 15703517.0
        AGRO          4  2722495.0


Looking at this table, a risk manager would feel reasonably comfortable. The NPL ratio sits near the ~4% benchmark for conservative Mexican institutional lenders (CNBV publishes these). Exposure is distributed across sectors. Individual and corporate loans split roughly 70/30. Nothing triggers an alarm.

**This is snapshot of the traditional analysis. Now using topological lens could reached something different**

The numbers above come from treating each loan as an independent row. But loans aren't independent -- they're connected through the people and companies behind them. The next section explores what those connections reveal.

---
## Section 2: What the Graph Reveals

I'm going to run four queries against the live graph. Each one finds a structural risk pattern that is invisible to flat-table analysis, not because SQL is bad, but because SQL has a clear schema design and well define purposes dimensionality if fixed from beginnig, and relationships are not first-class objects in a relational database.

Each finding connects to a failure mode from the 2008 financial crisis. These aren't rhetorical parallels, the crisis is literally why the data sources I used (GLEIF, beneficial ownership registries) were created.

### Finding 1: A "Safe" Client Who Isn't Safe

Here's something I wouldn't have thought to look for in a loan table: a client with an excellent credit score and zero past-due days who is nonetheless exposed to a live default.

The mechanism is a guarantee chain. Person A guarantees Person B's loan. Person B guarantees Person C's loan. Person C is in default. The risk flows backward through the chain like dominoes,but Person A's credit score knows nothing about Person C's existence.

In the 2008 crisis, AIG's guarantee chains on CDO tranches worked exactly this way. The depth of exposure was unknowable until the underlying asset failed. AIG had a AAA rating while sitting on top of a chain that collapsed entirely.

In [5]:
# Finding 1: The guarantee chain
chain = query("""
    MATCH (a:ClienteIndividual {id_cliente: 'CLI-00010'})
          -[g1:GARANTIZA]->(p1:Prestamo)
          <-[:TIENE_PRESTAMO]-(b:ClienteIndividual)
          -[g2:GARANTIZA]->(p2:Prestamo)
          <-[:TIENE_PRESTAMO]-(c:ClienteIndividual)
    WHERE p2.estatus = 'VENCIDO'
    RETURN a.nombre_completo AS hop_0_guarantor,
           a.score_buro AS hop_0_score,
           p1.id_prestamo AS hop_1_loan,
           b.nombre_completo AS hop_1_debtor,
           b.score_buro AS hop_1_score,
           p2.id_prestamo AS hop_2_loan,
           c.nombre_completo AS hop_2_defaulter,
           c.score_buro AS hop_2_score,
           p2.saldo_vigente_mxn AS exposed_amount,
           p2.dias_mora AS dias_mora
""")
print("=== GUARANTEE CHAIN (depth 3) ===")
print(chain.to_string(index=False))

if not chain.empty:
    print(f"\nThe guarantor at hop 0 has score {chain['hop_0_score'].iloc[0]} -- looks safe.")
    print(f"But they are 2 hops from a default of ${chain['exposed_amount'].iloc[0]:,.2f} MXN.")
    print(f"Their score responde to a different question nature. The graph does on other way.")

=== GUARANTEE CHAIN (depth 3) ===
     hop_0_guarantor  hop_0_score hop_1_loan          hop_1_debtor  hop_1_score hop_2_loan           hop_2_defaulter  hop_2_score  exposed_amount  dias_mora
Homero Aguayo Crespo        725.8  PRE-00127 Manuel Molina Ledesma        798.2  PRE-00254 Cecilia Galván Miramontes        649.6       132511.37         45

The guarantor at hop 0 has score 725.8 -- looks safe.
But they are 2 hops from a default of $132,511.37 MXN.
Their score responde to a different question nature. The graph does on other way.


### Finding 2: Coverage That Doesn't Exist

This one surprised me the most but a real simple relation too; a black swan discover. Three clients guarantee each other's loans in a closed loop. On paper, every loan has 100% guarantee coverage. A SQL query asking "does this loan have a guarantor?" would return YES for all three. A risk system would flag them as fully covered and flag as low priority.

But the coverage is fiction. If any one of them can't pay, the guarantee activates for the next person, who then can't cover their own guarantee, which activates the next one. The "protection" is circular; nobody outside the loop is backing anything. The total amount reported as "covered" is actually completely exposed. Kind of scam if were done delibertly

This is illegal under Mexican banking regulation (CNBV Circular 3/2012) because it creates fictitious coverage. And it's exactly what happened with AIG: they insured CDO tranches that were themselves backed by mortgages AIG had also insured. When the underlying failed, the "insurance" was worthless because the insurer was exposed to the same risk.

The critical point: **a SQL query cannot detect this.** Cycle detection at arbitrary depth requires recursive CTEs with cycle-check conditions that break above depth 3 in practice. In Cypher, it's one pattern match.

In [6]:
# Finding 2: Circular guarantee
cycle = query("""
    MATCH (c1:ClienteIndividual)-[:GARANTIZA]->(p1:Prestamo)
          <-[:TIENE_PRESTAMO]-(c2:ClienteIndividual)
          -[:GARANTIZA]->(p2:Prestamo)
          <-[:TIENE_PRESTAMO]-(c3:ClienteIndividual)
          -[:GARANTIZA]->(p3:Prestamo)
          <-[:TIENE_PRESTAMO]-(c1)
    WHERE c1 <> c2 AND c2 <> c3 AND c1 <> c3
    RETURN c1.nombre_completo AS person_1,
           p1.saldo_vigente_mxn AS loan_1_saldo,
           c2.nombre_completo AS person_2,
           p2.saldo_vigente_mxn AS loan_2_saldo,
           c3.nombre_completo AS person_3,
           p3.saldo_vigente_mxn AS loan_3_saldo
    LIMIT 1
""")
print("=== CIRCULAR GUARANTEE DETECTED ===")
print(cycle.to_string(index=False))

if not cycle.empty:
    total = cycle['loan_1_saldo'].iloc[0] + cycle['loan_2_saldo'].iloc[0] + cycle['loan_3_saldo'].iloc[0]
    print(f"\nTotal outstanding in cycle: ${total:,.2f} MXN")
    print(f"Reported coverage: 100% on all three loans")
    print(f"Actual coverage: 0% -- no external guarantor exists")
    print(f"Regulatory violation: CNBV Circular 3/2012")

=== CIRCULAR GUARANTEE DETECTED ===
            person_1  loan_1_saldo             person_2  loan_2_saldo                 person_3  loan_3_saldo
Juana Ledesma Rendón      50922.82 Soledad Gil Longoria        9200.9 Angélica Gonzales Campos      50852.01

Total outstanding in cycle: $110,975.73 MXN
Reported coverage: 100% on all three loans
Actual coverage: 0% -- no external guarantor exists
Regulatory violation: CNBV Circular 3/2012


### Finding 3: One Company Falls, Five People Follow

Now I want to look at a company that's already in trouble -- REESTRUCTURADO status -- and trace who else gets pulled down with it.

EMP-00031 has 5 shareholders. Each of them also has personal loans with the institution. Two of them cross-guarantee each other's personal loans. If the company fails completely, all 5 shareholders see their net worth drop simultaneously. The two cross-guarantors are doubly exposed: they lose equity AND they're on the hook for each other's personal debt.

Bear Stearns in 2008 wasn't the largest bank. It was the most interconnected. Its failure cascaded through counterparty relationships that nobody had mapped until it was too late. The phrase "too interconnected to fail" was coined for exactly this topology, where the SIZE of the entity doesn't predict the DAMAGE of its failure. The graph structure could detect it.

In [7]:
# Finding 3: Contagion hub
hub = query("""
    MATCH (e:Empresa {id_empresa: 'EMP-00031'})<-[:ES_ACCIONISTA_DE|ES_DIRECTOR_DE]-(c:ClienteIndividual)
    WITH e, c
    OPTIONAL MATCH (c)-[:TIENE_PRESTAMO]->(personal:Prestamo)
        WHERE personal.estatus IN ['ACTIVO', 'REESTRUCTURADO']
    OPTIONAL MATCH (c)-[:GARANTIZA]->(guaranteed:Prestamo)
        WHERE guaranteed.estatus IN ['ACTIVO', 'REESTRUCTURADO']
    RETURN c.nombre_completo AS shareholder,
           c.score_buro AS score,
           count(DISTINCT personal) AS personal_loans,
           round(sum(DISTINCT personal.saldo_vigente_mxn)) AS personal_exposure,
           count(DISTINCT guaranteed) AS guaranteed_loans,
           round(sum(DISTINCT guaranteed.saldo_vigente_mxn)) AS guarantee_exposure
""")
print("=== CONTAGION HUB: EMP-00031 ===")

# Company info
company = query("""
    MATCH (e:Empresa {id_empresa: 'EMP-00031'})-[:TIENE_PRESTAMO]->(p:Prestamo)
    RETURN e.razon_social AS empresa,
           p.estatus AS estatus,
           p.saldo_vigente_mxn AS saldo
""")
print(company.to_string(index=False))
print()
print("Shareholders exposed:")
print(hub.to_string(index=False))

if not hub.empty:
    total_personal = hub['personal_exposure'].sum()
    total_guarantee = hub['guarantee_exposure'].sum()
    print(f"\nTotal personal exposure of shareholders: ${total_personal:,.0f} MXN")
    print(f"Total guarantee exposure of shareholders: ${total_guarantee:,.0f} MXN")
    print(f"All of this becomes correlated if EMP-00031 fails.")

=== CONTAGION HUB: EMP-00031 ===
           empresa        estatus     saldo
HARI MASA SA DE CV REESTRUCTURADO 139536.52

Shareholders exposed:
                shareholder  score  personal_loans  personal_exposure  guaranteed_loans  guarantee_exposure
       Gabino Aparicio Mata  615.9               0                0.0                 5           2621267.0
        Laura Jasso Barraza  676.3               1           154168.0                 2            219429.0
Dolores Valverde Mascareñas  727.5               1            24165.0                 1            139537.0
     Judith Corral Gallegos  632.5               1            97286.0                 5           1737225.0
         Rubén Gamez Rivero  661.9               1            34912.0                 2            236822.0

Total personal exposure of shareholders: $310,531 MXN
Total guarantee exposure of shareholders: $4,954,280 MXN
All of this becomes correlated if EMP-00031 fails.


### Finding 4: Four Companies That Are Really One Risk

This is perhaps the most intuitive finding, but also the one most systematically missed by SQL-based reporting.

One person controls four separate companies. In the loan table, these are four independent borrowers with four separate risk assessments. The portfolio report shows exposure spread across four entities in different sectors. It looks diversified.

The graph shows it's one person. If that person has a health crisis, a legal problem, or simply makes a bad strategic decision, all four companies lose their key decision-maker in the same moment. The "diversified" exposure concentrates into a single correlated event.

Lehman Brothers had 7,000 legal entities across 40 countries. The portfolio looked maximally diversified. But one controlling group sat behind all of them. When the parent failed, all 7,000 failed simultaneously. This is literally why the G20 mandated the creation of GLEIF in 2011, because regulators couldn't see through the entity layer to the control layer during the crisis.

Mexican regulation recognizes this through the concept of "personas relacionadas" (CNBV Circular Unica de Bancos, Article 73), which requires institutions to monitor and report concentration through shared directors and controlling shareholders. The graph makes this monitoring a one-line query instead of a manual review process.

In [8]:
# Finding 4: Hidden concentration
concentration = query("""
    MATCH (c:ClienteIndividual {id_cliente: 'CLI-00298'})
          -[r:ES_ACCIONISTA_DE]->(e:Empresa)
          -[:TIENE_PRESTAMO]->(p:Prestamo)
    WHERE p.estatus IN ['ACTIVO', 'REESTRUCTURADO']
    RETURN e.razon_social AS empresa,
           e.sector AS sector,
           r.porcentaje_participacion AS ownership,
           p.id_prestamo AS prestamo,
           p.saldo_vigente_mxn AS saldo,
           p.estatus AS estatus
    ORDER BY saldo DESC
""")
print("=== HIDDEN CONCENTRATION ===")
print(f"Controller: CLI-00298")

# Get the person's name and personal loans
person = query("""
    MATCH (c:ClienteIndividual {id_cliente: 'CLI-00298'})
    OPTIONAL MATCH (c)-[:TIENE_PRESTAMO]->(p:Prestamo)
        WHERE p.estatus IN ['ACTIVO', 'REESTRUCTURADO']
    RETURN c.nombre_completo AS nombre,
           c.score_buro AS score,
           c.nivel_ingresos AS ingresos,
           count(p) AS personal_loans,
           round(sum(p.saldo_vigente_mxn)) AS personal_exposure
""")
print(person.to_string(index=False))
print()
print("Companies controlled:")
print(concentration.to_string(index=False))

if not concentration.empty and not person.empty:
    corp_total = concentration['saldo'].sum()
    personal_total = person['personal_exposure'].iloc[0] or 0
    n_companies = concentration['empresa'].nunique()
    print(f"\nSQL sees: {n_companies} separate companies (diversified)")
    print(f"Graph sees: 1 person controlling all {n_companies}")
    print(f"Corporate exposure through this person: ${corp_total:,.2f} MXN")
    print(f"Personal exposure: ${personal_total:,.0f} MXN")
    print(f"TRUE concentration: ${corp_total + personal_total:,.0f} MXN")

=== HIDDEN CONCENTRATION ===
Controller: CLI-00298


                nombre  score  ingresos  personal_loans  personal_exposure
Emilio Maldonado Yáñez  583.3  18784.24               1           119144.0

Companies controlled:
                               empresa      sector  ownership  prestamo      saldo        estatus
                   CORPORATIVO FREMING    COMERCIO       0.14 PRE-00449 3218957.65 REESTRUCTURADO
                  MAQUILACERO SA DE CV MANUFACTURA       0.94 PRE-00392 1125686.11 REESTRUCTURADO
M. HOLLAND LATINOAMERICA S DE RL DE CV   SERVICIOS       0.27 PRE-00448  881614.55         ACTIVO
                  MAQUILACERO SA DE CV MANUFACTURA       0.94 PRE-00451  635225.73         ACTIVO
                   CORPORATIVO FREMING    COMERCIO       0.14 PRE-00369  624473.28         ACTIVO
   FUERZA Y ENERGIA BII HIOXO SA DE CV    COMERCIO       0.63 PRE-00342  395293.69         ACTIVO
   FUERZA Y ENERGIA BII HIOXO SA DE CV    COMERCIO       0.63 PRE-00422  125011.26         ACTIVO
M. HOLLAND LATINOAMERICA S DE RL DE CV   SE

---
## Section 3: What This Means for Provisioning

Here's where I want to connect these structural findings back to a number the risk fecision maker thinks about.

A standard provisioning model calculates expected loss as PD * EAD for each loan and sums across the portfolio. This assumes independence, and then zero correlation, each loan's default probability has nothing to do with any other loan's.

But the graph just showed us that this assumption is wrong in at least four ways:
- Guarantee chains make "safe" clients indirectly exposed to defaults
- Circular guarantees inflate reported coverage from 100% to effectively 0%
- Corporate hubs create correlated failure events across multiple shareholders
- Entity-level diversification masks person-level concentration

The naive calculation UNDERSTATES the true expected loss. Let explore differences.

In [9]:
# Portfolio-level impact: what SQL misses

# 1. Total exposure in guarantee chains (not visible in flat tables)
chain_exposure = query("""
    MATCH (garante:ClienteIndividual)-[:GARANTIZA]->(p:Prestamo)
    WHERE p.estatus IN ['ACTIVO', 'REESTRUCTURADO', 'VENCIDO']
    RETURN round(sum(p.saldo_vigente_mxn)) AS total_guaranteed_exposure,
           count(DISTINCT garante) AS num_guarantors,
           count(p) AS num_guaranteed_loans
""")
print("=== INDIRECT EXPOSURE (invisible to SQL) ===")
print(chain_exposure.to_string(index=False))

# 2. Unsecured active loans (no guarantor at all)
unsecured = query("""
    MATCH (p:Prestamo {estatus: 'ACTIVO'})
    WHERE NOT EXISTS { MATCH ()-[:GARANTIZA]->(p) }
    RETURN count(p) AS unsecured_loans,
           round(sum(p.saldo_vigente_mxn)) AS unsecured_exposure
""")
print("\n=== UNSECURED ACTIVE LOANS ===")
print(unsecured.to_string(index=False))

# 3. Circular guarantee exposure (fictitious coverage)
circular = query("""
    MATCH (c1:ClienteIndividual)-[:GARANTIZA]->(p1:Prestamo)
          <-[:TIENE_PRESTAMO]-(c2:ClienteIndividual)
          -[:GARANTIZA]->(p2:Prestamo)
          <-[:TIENE_PRESTAMO]-(c3:ClienteIndividual)
          -[:GARANTIZA]->(p3:Prestamo)
          <-[:TIENE_PRESTAMO]-(c1)
    WHERE c1 <> c2 AND c2 <> c3 AND c1 <> c3
    RETURN round(p1.saldo_vigente_mxn + p2.saldo_vigente_mxn + p3.saldo_vigente_mxn) AS circular_exposure
    LIMIT 1
""")
print("\n=== CIRCULAR GUARANTEE EXPOSURE (fictitious coverage) ===")
print(circular.to_string(index=False))

# 4. Concentration risk: persons controlling 2+ companies
concentration_risk = query("""
    MATCH (c:ClienteIndividual)-[:ES_ACCIONISTA_DE|ES_DIRECTOR_DE]->(e:Empresa)
    WITH c, count(e) AS num_companies, collect(e.id_empresa) AS companies
    WHERE num_companies >= 2
    OPTIONAL MATCH (e2:Empresa)-[:TIENE_PRESTAMO]->(p:Prestamo)
        WHERE e2.id_empresa IN companies
        AND p.estatus IN ['ACTIVO', 'REESTRUCTURADO']
    RETURN c.nombre_completo AS controller,
           num_companies,
           round(sum(p.saldo_vigente_mxn)) AS concentrated_exposure
    ORDER BY concentrated_exposure DESC
""")
print("\n=== CONCENTRATION RISK (multi-company controllers) ===")
print(concentration_risk.to_string(index=False))

=== INDIRECT EXPOSURE (invisible to SQL) ===
 total_guaranteed_exposure  num_guarantors  num_guaranteed_loans
               119703023.0             131                   178

=== UNSECURED ACTIVE LOANS ===
 unsecured_loans  unsecured_exposure
             198          79001166.0



=== CIRCULAR GUARANTEE EXPOSURE (fictitious coverage) ===
 circular_exposure
          110976.0



=== CONCENTRATION RISK (multi-company controllers) ===
             controller  num_companies  concentrated_exposure
 Emilio Maldonado Yáñez              4              7044416.0
    Ernesto Vera Jaimes              2              4018033.0
Natalia Centeno Vásquez              2              3862322.0
 Judith Corral Gallegos              2              2967641.0
   Gabino Aparicio Mata              2              2102433.0
     Mónica Zepeda Piña              2              1537805.0
     Rubén Gamez Rivero              2               139537.0


In [10]:
# Summary: the gap between SQL view and graph view
print("=" * 60)
print("SUMMARY: SQL VIEW vs GRAPH VIEW")
print("=" * 60)
print()
print("What SQL reports:")
print("  - Portfolio is diversified across individual and corporate loans")
print("  - NPL ratio is within Mexican conservative lender benchmarks")
print("  - All guaranteed loans show 100% coverage")
print()
print("What the graph reveals:")
print("  1. Guarantee chains create hidden indirect exposure")
print("  2. Circular guarantees inflate reported coverage (CNBV violation)")
print("  3. Corporate contagion hubs concentrate risk in single-failure events")
print("  4. Entity-level diversification masks person-level concentration")
print()
print("The method works. Swap synthetic data for a real loan book,")
print("keep the graph schema and queries, and it runs unchanged.")
print("The difference between this prototype and production is")
print("scale and data source, an volume of complex and not trivial")
print("relation not logic.")

SUMMARY: SQL VIEW vs GRAPH VIEW

What SQL reports:
  - Portfolio is diversified across individual and corporate loans
  - NPL ratio is within Mexican conservative lender benchmarks
  - All guaranteed loans show 100% coverage

What the graph reveals:
  1. Guarantee chains create hidden indirect exposure
  2. Circular guarantees inflate reported coverage (CNBV violation)
  3. Corporate contagion hubs concentrate risk in single-failure events
  4. Entity-level diversification masks person-level concentration

The method works. Swap synthetic data for a real loan book,
keep the graph schema and queries, and it runs unchanged.
The difference between this prototype and production is
scale and data source, an volume of complex and not trivial
relation not logic.


---
## What I Learned

Building this project changed how I think about credit risk. Not because the graph paradigm is universally better than SQL. SQL is the right tool for most portfolio queries and is a universal structure data that works. But for a specific class of questions, the ones about connections, chains, cycles, and contagion, different natures attributes, the relational model doesn't just struggle. It structurally cannot express the question.

The 2008 crisis didn't happen because banks had bad data. It happened because the data was organized in a way that made systemic structure invisible. Lehman's 7,000 entities were all in the database. AIG's CDO guarantees were all documented. The information existed. But the paradigm for analysis, tables, JOINs, entity-level aggregation, couldn't surface the patterns that mattered.

Graph databases don't prevent crises. But they make the questions askable. And that's the prerequisite for everything else.

In [11]:
# Clean up
driver.close()
print("Connection closed")

Connection closed
